# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AW-OMW/FLY-RANK-PROJECT/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [12]:
selected_features = [
    'search_volume', 'competition', 'competition_level', 'cpc',
    'content_type', 'main_intent', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
    'users_90d', 'engaged_sessions_90d', 'scroll_events_90d',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate',
    'ai_traffic_pct', 'days_since_last_update', 'freshness_tier',
    'trend_direction', 'trend_pct'
]

# Display a sample row with these selected features
display(df[selected_features].sample(1))

,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,impressions_90d,clicks_90d,...,sessions_last_30d,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,days_since_last_update,freshness_tier,trend_direction,trend_pct
10615,0.0,0.0,LOW,0.0,keyword article,informational,2877.0,21542.0,35,0,...,2,0.0,7.0,0.0,200.0,0.0,20,0-30,down,-64.7


In [13]:
#over the dates 1 april 2026 to 30 june 2026 the data shows the page is in decline and no potential chance of boom

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [14]:
# Check for duplicate 'content_id' rows
duplicate_content_ids = df['content_id'].value_counts()
duplicate_content_ids = duplicate_content_ids[duplicate_content_ids > 1]

if not duplicate_content_ids.empty:
    print("Content IDs appearing more than once (indicating multiple monthly observations):")
    display(duplicate_content_ids.head())
    print(f"Total unique content IDs with multiple entries: {len(duplicate_content_ids)}")
    print(f"Max occurrences for a single content ID: {duplicate_content_ids.max()}")
else:
    print("No duplicate content_id rows found. Each content_id appears exactly once.")




No duplicate content_id rows found. Each content_id appears exactly once.


In [15]:
# Total number of rows
print(f"Total number of rows in the DataFrame: {len(df)}")

# Number of unique content_ids (should be equal to total rows based on previous check)
print(f"Number of unique content IDs: {df['content_id'].nunique()}")

Total number of rows in the DataFrame: 30000
Number of unique content IDs: 30000


### Descriptive Statistics for Numerical Features

This will give you counts, means, standard deviations, min/max values, and quartiles for all numerical columns.

In [16]:
display(df.describe())

,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,sessions_prev_30d,content_age_days,age_tier_order,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,trend_pct
count,27532.000000,27532.000000,27532.000000,22301.000000,22301.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.00000,30000.000000,30000.000000,30000.000000,30000.00000,30000.000000,29875.000000,30000.000000,26612.000000
mean,158.882391,0.146954,0.485342,3107.760325,20665.277835,5200.366300,16.097333,49.942467,37.066633,35.937700,...,10.283000,256.16780,4.786533,46.098300,0.510733,16.34238,2.534520,18.212921,0.768196,-4.785969
std,1518.270825,0.285241,2.101560,1452.382598,10115.344042,16838.019547,75.076958,152.101430,107.069131,103.748185,...,42.578003,132.70793,0.790392,42.078709,3.279162,15.21679,8.310096,29.472768,7.429454,473.861780
min,0.000000,0.000000,0.000000,8.000000,40.000000,1.000000,0.000000,0.000000,1.000000,1.000000,...,0.000000,90.00000,3.000000,1.000000,0.000000,0.00000,0.000000,0.000000,0.000000,-100.000000
25%,0.000000,0.000000,0.000000,2413.000000,15644.000000,81.000000,0.000000,2.000000,2.000000,2.000000,...,1.000000,132.00000,4.000000,20.000000,0.000000,6.20000,0.000000,0.000000,0.000000,-62.600000
50%,10.000000,0.000000,0.000000,2877.000000,19116.000000,731.000000,1.000000,8.000000,7.000000,7.000000,...,2.000000,236.00000,5.000000,20.000000,0.070000,10.80000,0.000000,5.000000,0.000000,-33.500000
75%,20.000000,0.130000,0.000000,3666.000000,24011.000000,3615.250000,7.000000,33.000000,27.000000,27.000000,...,7.000000,333.00000,5.000000,104.000000,0.290000,22.30000,1.350000,23.530000,0.000000,0.000000
max,74000.000000,1.000000,100.360000,9546.000000,111158.000000,517715.000000,4178.000000,5998.000000,4345.000000,4913.000000,...,4247.000000,564.00000,6.000000,373.000000,100.000000,245.00000,100.000000,300.000000,300.000000,44900.000000


### Counts for Categorical Features

Let's check the unique values and their counts for some of the categorical columns you've identified as features or context.

In [17]:
# Example for 'content_type'
if 'content_type' in df.columns:
    print("\nUnique counts for 'content_type':")
    display(df['content_type'].value_counts())

# Example for 'main_intent'
if 'main_intent' in df.columns:
    print("\nUnique counts for 'main_intent':")
    display(df['main_intent'].value_counts())

# Example for 'competition_level'
if 'competition_level' in df.columns:
    print("\nUnique counts for 'competition_level':")
    display(df['competition_level'].value_counts())


Unique counts for 'content_type':


,count
content_type,
keyword article,27207
feedly article,2096
comparison article,697



Unique counts for 'main_intent':


,count
main_intent,
informational,17235
transactional,5733
commercial,4612
navigational,46



Unique counts for 'competition_level':


,count
competition_level,
LOW,22896
HIGH,2658
MEDIUM,1836


In [18]:
print("\nMissing values per column:")
display(df.isnull().sum())


Missing values per column:


,0
content_id,0
client_id,0
search_volume,2468
competition,2468
competition_level,2610
cpc,2468
content_type,0
main_intent,2374
word_count,7699
char_count,7699


In [19]:
print("DataFrame Info:")
df.info()

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d         

From `df.info()`, we don't see an explicit `datetime` column. However, we have a `days_since_last_update` column, which is crucial for understanding the recency of the data. Let's examine its distribution.

In [20]:
if 'days_since_last_update' in df.columns:
    print("\nDescriptive statistics for 'days_since_last_update':")
    display(df['days_since_last_update'].describe())
else:
    print("'days_since_last_update' column not found.")


Descriptive statistics for 'days_since_last_update':


,days_since_last_update
count,30000.000000
mean,46.098300
std,42.078709
min,1.000000
25%,20.000000
50%,20.000000
75%,104.000000
max,373.000000


The `days_since_last_update` column provides insight into the recency of each content page. The descriptive statistics above will show the minimum, maximum, and average number of days since a page was last updated. Combined with other time-aggregated features like `_90d` and `_last_30d`, this suggests that the data likely represents a snapshot of performance leading up to a certain point in time, with varying degrees of recency for individual content pieces based on their last update.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
#this data cannot tell me exact calender date of data collection , also it cannot tell me historical trends because of each content id only appearing once

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.